<a href="https://colab.research.google.com/github/Alfonso-Jesus-Garcia-Moya/LINGUISTICA_COMPUTACIONAL/blob/SESION-7-An%C3%A1lisis-del-efecto-de-las-stopwords-en-el-procesamiento-de-texto/Sesion_7_An%C3%A1lisis_del_efecto_de_las_stopwords_en_el_procesamiento_de_texto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Thu Sep 25 17:18:19 2025

@author: julia
"""

# ======================================
# 1. Importar librerías
# ======================================
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report


# ======================================
# 2. Crear dataset ampliado y balanceado
# 0 = No irónico | 1 = Irónico
# ======================================
data = {
    "texto": [
        # Frases irónicas
        "El servicio fue excelente, me dejaron esperando tres horas",
        "Qué rápido llegó mi pedido, solo tardó una semana",
        "El celular es increíble, se apaga cada cinco minutos",
        "Qué maravilla, otra vez sin internet en pleno lunes",
        "El profesor es un genio, canceló la clase de nuevo",
        "Excelente gestión, perdimos todos los archivos",

        # Frases no irónicas
        "La atención al cliente fue muy buena y me resolvieron rápido",
        "El producto funciona perfecto, tal como esperaba",
        "Me encantó, llegó antes de lo previsto",
        "La aplicación funciona sin problemas, muy fluida",
        "El repartidor llegó puntual y fue muy amable",
        "El software funciona bien y es fácil de usar"
    ],
    "etiqueta": [1,1,1,1,1,1, 0,0,0,0,0,0]
}

df = pd.DataFrame(data)

# ======================================
# 3. Preparar los datos
# ======================================
X = df["texto"]
y = df["etiqueta"]

# Convertir texto en vectores TF-IDF
vectorizador = TfidfVectorizer()
X_tfidf = vectorizador.fit_transform(X)

# Dividir en entrenamiento y prueba (estratificado)
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.3, random_state=42, stratify=y
)

# ======================================
# 4. Entrenar el modelo
# ======================================
modelo = MultinomialNB()
modelo.fit(X_train, y_train)

# ======================================
# 5. Evaluar el modelo
# ======================================
y_pred = modelo.predict(X_test)
print("=== Reporte de clasificación ===")
print(classification_report(y_test, y_pred, zero_division=0))

# ======================================
# 6. Probar con nuevos textos
# ======================================
nuevos_textos = [
    "Qué maravilloso servicio, me ignoraron todo el tiempo",
    "El envío fue rápido y el producto llegó en buen estado",
    "El sistema es tan confiable que se cae cada cinco minutos"
]

nuevos_vectores = vectorizador.transform(nuevos_textos)
predicciones = modelo.predict(nuevos_vectores)

# Mostrar resultados
print("\n=== Predicciones con nuevos textos ===")
for texto, pred in zip(nuevos_textos, predicciones):
    etiqueta = "Irónico" if pred == 1 else "No irónico"
    print(f"Texto: {texto} → {etiqueta}")

=== Reporte de clasificación ===
              precision    recall  f1-score   support

           0       0.67      1.00      0.80         2
           1       1.00      0.50      0.67         2

    accuracy                           0.75         4
   macro avg       0.83      0.75      0.73         4
weighted avg       0.83      0.75      0.73         4


=== Predicciones con nuevos textos ===
Texto: Qué maravilloso servicio, me ignoraron todo el tiempo → Irónico
Texto: El envío fue rápido y el producto llegó en buen estado → No irónico
Texto: El sistema es tan confiable que se cae cada cinco minutos → Irónico


#Analisis de ALICIA_SPA CON STOPWORDS

In [18]:
#Se importan las librerias a usar
import spacy
import operator
import os
import math

nlp = spacy.load('es_core_news_md') #Se invoca el modelo de spacy

# --- INICIO DE LA MODIFICACIÓN ---

# 1. Indicamos la ruta del archivo txt en Colab
ruta_del_texto = "/content/alicia_spa.txt"

# 2. Creamos una lista con la ruta del archivo que quieres analizar
file_list = [ruta_del_texto]

total_documentos = len(file_list)
print('Número de documentos a analizar:', total_documentos)

# --- FIN DE LA MODIFICACIÓN ---

#Declaración de varibles globales
sortedDict= {}
contador = 1
documento_palabra = []
Total_TF =[]

#Fragmento de codigo para toquenizar y contar las palabras de cada archivo del corpus
for file_path in file_list:
    print(f"\n--- Analizando el archivo: {file_path} ---")
    tokens = []

    # --- MODIFICACIÓN DENTRO DEL BUCLE ---
    # Usamos la nueva ruta y añadimos encoding='utf-8' por si hay caracteres especiales
    # doc = open(file_path, 'rt', encoding='utf-8') # Original line
    doc = open(file_path, 'rt', encoding='latin-1') # Trying a different encoding
    datos = doc.read()
    datos = datos.lower()

    quitar = ",;:.\n¡¿?!«»\"'-––"
    for caracter in quitar:
      datos = datos.replace(caracter, "")
    result = nlp(datos)
    for token in result:
      #if token.pos_== "VERB" or token.pos_== "NOUN" or token.pos_== "ADJ" or token.pos_== "ADV":
        tokens.append(token.text)

    diccionario_frecuencias = {}
    for palabra in tokens:
      if palabra in diccionario_frecuencias:
        diccionario_frecuencias[palabra] += 1
      else:
        diccionario_frecuencias[palabra] = 1

    sortedDict = sorted(diccionario_frecuencias.items(), key=operator.itemgetter(1), reverse = True)

    # Esta línea ahora solo añadirá la palabra más frecuente de "alicia_spa.txt"
    if sortedDict:
        documento_palabra.append(sortedDict[0][0])

    print("Primeras 20 palabras más frecuentes:")
    for k,v in sortedDict[:20]:
        print(f'Palabra: *{k}* : {v} veces repetida')

    palabras = datos.split()
    total_palabras= len(palabras)
    print('\nNúmero de palabras en el archivo de texto:', total_palabras)

    # El resto del código de TF-IDF se ejecutará, pero ten en cuenta la siguiente nota.
    # ... (el resto del script continúa igual)
    # ...

Número de documentos a analizar: 1

--- Analizando el archivo: /content/alicia_spa.txt ---
Primeras 20 palabras más frecuentes:
Palabra: *de* : 1077 veces repetida
Palabra: *que* : 964 veces repetida
Palabra: *la* : 918 veces repetida
Palabra: *y* : 817 veces repetida
Palabra: *el* : 707 veces repetida
Palabra: *a* : 704 veces repetida
Palabra: *en* : 463 veces repetida
Palabra: *se* : 462 veces repetida
Palabra: *no* : 424 veces repetida
Palabra: *un* : 345 veces repetida
Palabra: *alicia* : 317 veces repetida
Palabra: *con* : 311 veces repetida
Palabra: *lo* : 298 veces repetida
Palabra: *dijo* : 258 veces repetida
Palabra: *los* : 250 veces repetida
Palabra: *una* : 245 veces repetida
Palabra: *por* : 201 veces repetida
Palabra: *al* : 181 veces repetida
Palabra: *le* : 173 veces repetida
Palabra: *del* : 153 veces repetida

Número de palabras en el archivo de texto: 26074


#Analisis de ALICIA_SPA SIN STOPWORDS